In [1]:
# [code]
# 1. Import Library dan Konfigurasi Supabase
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from supabase import create_client, Client
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta

# Muat variabel lingkungan dari file .env
load_dotenv()

# --- Konfigurasi Supabase ---
# Ganti dengan nama tabel dan kolom Anda di Supabase
TABLE_NAME = "tb_konsentrasi_gas" 
COL_PM25 = "pm25_ugm3"
COL_NO= "no2_ugm3"
COL_PM10 = "pm10_ugm3"
COL_CO = "co_ugm3"
COL_SUHU = "temperature"
COL_KELEMBAPAN = "humidity"
COL_TIMESTAMP = "created_at"

# Ambil URL dan Key dari environment variables
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Pastikan SUPABASE_URL dan SUPABASE_KEY sudah diatur di file .env")

# Inisialisasi klien Supabase
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Langkah 1: Lingkungan dan Supabase berhasil dikonfigurasi.")

Langkah 1: Lingkungan dan Supabase berhasil dikonfigurasi.


In [2]:
# [code]
# 2. Mengambil Data dari Tabel 'tb_konsentrasi_gas'
from datetime import datetime, timedelta
# Tentukan rentang waktu: 7 hari yang lalu hingga sekarang
seven_days_ago = datetime.now() - timedelta(days=14)
print(f"Langkah 2: Mengambil data dari {seven_days_ago.strftime('%Y-%m-%d %H:%M')}...")

# --- KODE PAGINATION ---
all_data = []
page_size = 1000
offset = 0

# Ganti nama kolom sesuai dengan yang ada di database Anda
# Menggunakan nama yang Anda berikan di kode sebelumnya
COL_PM25 = "pm25_ugm3"
COL_NO= "no2_ugm3"
COL_PM10 = "pm10_ugm3"
COL_CO = "co_ugm3"
COL_SUHU = "temperature"
COL_KELEMBAPAN = "humidity"
COL_TIMESTAMP = "created_at"

while True:
    print(f"Mengambil {page_size} data dimulai dari baris ke-{offset}...")
    
    response = supabase.table(TABLE_NAME) \
        .select(f"{COL_PM25},{COL_PM10}, {COL_NO}, {COL_CO}, {COL_SUHU}, {COL_KELEMBAPAN}, {COL_TIMESTAMP}") \
        .gte(COL_TIMESTAMP, seven_days_ago.isoformat()) \
        .order(COL_TIMESTAMP, desc=False) \
        .range(offset, offset + page_size - 1) \
        .execute()
    
    current_data = response.data
    if not current_data:
        print("Semua data telah diambil.")
        break
        
    all_data.extend(current_data)
    offset += len(current_data)

# Konversi ke DataFrame dan lakukan pembersihan
df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    print("ERROR: Tidak ada data yang ditemukan.")
else:
    print(f"\nData mentah berhasil diambil. Total baris: {len(df_raw)}")
    
    # Konversi semua kolom ke numerik
    for col in [COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN]:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    
    df_raw[COL_TIMESTAMP] = pd.to_datetime(df_raw[COL_TIMESTAMP])

    # Hapus baris dengan nilai kosong
    initial_count = len(df_raw)
    df_raw.dropna(subset=[COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN], inplace=True)
    final_count = len(df_raw)
    print(f"{initial_count - final_count} baris dengan nilai tidak valid telah dihapus.")

    # Bersihkan dan siapkan data
    df_clean = df_raw.set_index(COL_TIMESTAMP)
    df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
    df_clean.sort_index(inplace=True)
    df_clean.rename(columns={
        COL_PM25: 'pm2.5', 
        COL_PM10: "pm10_corrected_ugm3",
        COL_NO: "no2_ugm3",
        COL_CO: "co_ugm3",
        COL_SUHU: "temperature",
        COL_KELEMBAPAN: "humidity"
    }, inplace=True)
    
    print("\nData yang bersih siap digunakan.")
    print("5 baris pertama:")
    print(df_clean.head())

Langkah 2: Mengambil data dari 2026-03-26 19:10...
Mengambil 1000 data dimulai dari baris ke-0...
Mengambil 1000 data dimulai dari baris ke-1000...
Mengambil 1000 data dimulai dari baris ke-2000...
Mengambil 1000 data dimulai dari baris ke-3000...
Mengambil 1000 data dimulai dari baris ke-4000...
Mengambil 1000 data dimulai dari baris ke-5000...
Mengambil 1000 data dimulai dari baris ke-6000...
Mengambil 1000 data dimulai dari baris ke-7000...
Mengambil 1000 data dimulai dari baris ke-8000...
Mengambil 1000 data dimulai dari baris ke-9000...
Mengambil 1000 data dimulai dari baris ke-10000...
Mengambil 1000 data dimulai dari baris ke-11000...
Mengambil 1000 data dimulai dari baris ke-11647...
Semua data telah diambil.

Data mentah berhasil diambil. Total baris: 11647
0 baris dengan nilai tidak valid telah dihapus.

Data yang bersih siap digunakan.
5 baris pertama:
                                  pm2.5  pm10_corrected_ugm3  no2_ugm3  \
created_at                                        

In [3]:
# [code]
# 1. Mengambil Data Per Menit (7 Hari Terakhir)

from datetime import datetime, timedelta

# Kita mulai dengan 7 hari data per menit
seven_days_ago = datetime.now() - timedelta(days=14)
print(f"Langkah 1: Mengambil data per menit dari {seven_days_ago.strftime('%Y-%m-%d %H:%M')}...")

# --- KODE PAGINATION ---
all_data = []
page_size = 1000
offset = 0

# Ganti nama kolom sesuai dengan yang ada di database Anda
TABLE_NAME = "tb_konsentrasi_gas" 
COL_PM25 = "pm25_ugm3"
COL_PM10= "pm10_ugm3"
COL_NO= "no2_ugm3"
COL_CO= "co_ugm3"
COL_SUHU= "temperature"
COL_KELEMBAPAN= "humidity"
while True:
    print(f"Mengambil {page_size} data dimulai dari baris ke-{offset}...")
    
    response = supabase.table(TABLE_NAME) \
        .select(f"{COL_PM25},{COL_PM10}, {COL_NO}, {COL_CO}, {COL_SUHU}, {COL_KELEMBAPAN}, {COL_TIMESTAMP}") \
        .gte(COL_TIMESTAMP, seven_days_ago.isoformat()) \
        .order(COL_TIMESTAMP, desc=False) \
        .range(offset, offset + page_size - 1) \
        .execute()
    
    current_data = response.data
    if not current_data:
        print("Semua data telah diambil.")
        break
        
    all_data.extend(current_data)
    offset += len(current_data)

# Konversi ke DataFrame dan lakukan pembersihan
df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    print("ERROR: Tidak ada data yang ditemukan.")
else:
    print(f"\nData mentah berhasil diambil. Total baris: {len(df_raw)}")
    
    # Konversi semua kolom ke numerik
    for col in [COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN]:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    
    df_raw[COL_TIMESTAMP] = pd.to_datetime(df_raw[COL_TIMESTAMP])

    # Hapus baris dengan nilai kosong
    initial_count = len(df_raw)
    df_raw.dropna(subset=[COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN], inplace=True)
    final_count = len(df_raw)
    print(f"{initial_count - final_count} baris dengan nilai tidak valid telah dihapus.")

    # Bersihkan dan siapkan data per menit
    df_clean = df_raw.set_index(COL_TIMESTAMP)
    df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
    df_clean.sort_index(inplace=True)
    df_clean.rename(columns={
        COL_PM25: 'pm2.5', 
        COL_PM10: "pm10_ugm3",
        COL_NO: "no2_ugm3",
        COL_CO: "co_ugm3",
        COL_SUHU: "temperature",
        COL_KELEMBAPAN: "humidity"
    }, inplace=True)
    print(f"Data per menit yang bersih telah disiapkan. Total baris: {len(df_clean)}")

Langkah 1: Mengambil data per menit dari 2026-03-26 19:10...
Mengambil 1000 data dimulai dari baris ke-0...
Mengambil 1000 data dimulai dari baris ke-1000...
Mengambil 1000 data dimulai dari baris ke-2000...
Mengambil 1000 data dimulai dari baris ke-3000...
Mengambil 1000 data dimulai dari baris ke-4000...
Mengambil 1000 data dimulai dari baris ke-5000...
Mengambil 1000 data dimulai dari baris ke-6000...
Mengambil 1000 data dimulai dari baris ke-7000...
Mengambil 1000 data dimulai dari baris ke-8000...
Mengambil 1000 data dimulai dari baris ke-9000...
Mengambil 1000 data dimulai dari baris ke-10000...
Mengambil 1000 data dimulai dari baris ke-11000...
Mengambil 1000 data dimulai dari baris ke-11647...
Semua data telah diambil.

Data mentah berhasil diambil. Total baris: 11647
0 baris dengan nilai tidak valid telah dihapus.
Data per menit yang bersih telah disiapkan. Total baris: 11647


In [4]:
# [code]
# 2. Rekayasa Fitur untuk Prediksi 1 Menit ke Depan

if 'df_clean' in locals() and not df_clean.empty:
    print("Langkah 2: Membuat fitur dari data per menit...")
    
    # --- PERUBAHAN UTAMA: TIDAK ADA RESAMPLING ---
    # Kita bekerja langsung dengan df_clean yang berindeks per menit
    features_df = df_clean.copy()
    feature_cols = ['pm2.5', "pm10_ugm3",
    "co_ugm3",
    "no2_ugm3",
    "temperature",
    "humidity"]
    
    # Buat fitur lag dan rolling yang sesuai untuk skala menit
    for col in feature_cols:
        # Fitur Lag (jeda waktu)
        features_df[f'{col}_lag_1min'] = features_df[col].shift(1)
        features_df[f'{col}_lag_5min'] = features_df[col].shift(5)
        features_df[f'{col}_lag_15min'] = features_df[col].shift(15)
        features_df[f'{col}_lag_60min'] = features_df[col].shift(60) # 1 jam lalu

        # Fitur Rolling Window (jendela geser)
        features_df[f'{col}_rolling_mean_5min'] = features_df[col].rolling(window=5).mean()
        features_df[f'{col}_rolling_std_5min'] = features_df[col].rolling(window=5).std()
        features_df[f'{col}_rolling_mean_15min'] = features_df[col].rolling(window=15).mean()

    # Fitur Berbasis Waktu (tambahkan 'minute')
    features_df['minute'] = features_df.index.minute
    features_df['hour'] = features_df.index.hour
    features_df['dayofweek'] = features_df.index.dayofweek

    # Tentukan Target (y): nilai di 1 menit ke depan
    features_df['target'] = features_df['pm2.5'].shift(-1)
    
    # Hapus baris dengan nilai NaN
    features_df.dropna(inplace=True)

    print(f"\nJumlah baris akhir setelah pembuatan fitur: {len(features_df)}")
    print(f"Jumlah total fitur: {len(features_df.columns)}")
    
    if features_df.empty:
        print("ERROR: DataFrame fitur kosong.")
    else:
        print("\n5 baris pertama dari DataFrame fitur:")
        print(features_df.head())
else:
    print("Langkah 2 dilewati.")

Langkah 2: Membuat fitur dari data per menit...

Jumlah baris akhir setelah pembuatan fitur: 11586
Jumlah total fitur: 52

5 baris pertama dari DataFrame fitur:
                                   pm2.5  pm10_ugm3  no2_ugm3   co_ugm3  \
created_at                                                                
2026-03-30 13:26:59.234565+00:00  128.74     164.75       0.0  59283.37   
2026-03-30 13:28:00.052899+00:00   99.66     127.11       0.0  58118.74   
2026-03-30 13:29:01.134223+00:00  126.80     166.37       0.0  58872.32   
2026-03-30 13:30:02.176204+00:00  109.53     154.76       0.0  59351.87   
2026-03-30 13:31:02.891395+00:00  135.61     165.45       0.0  59077.84   

                                  temperature  humidity  pm2.5_lag_1min  \
created_at                                                                
2026-03-30 13:26:59.234565+00:00         32.4      83.1          132.44   
2026-03-30 13:28:00.052899+00:00         32.4      83.9          128.74   
2026-03-30 13

In [5]:
# =========================
# ADVANCED OPTUNA TUNING for LightGBM (Time-Series CV)
# =========================
import os
import math
import joblib
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime
from optuna.exceptions import TrialPruned
import warnings
warnings.filterwarnings("ignore")

# -------------------------
# CONFIG: sesuaikan kalau perlu
# -------------------------
TARGET_COL = "pm2.5"        # nama kolom target di features_df
TEST_RATIO = 0.7           # porsi untuk holdout terakhir
N_FOLDS = 5                # jumlah fold rolling-origin
N_TRIALS = 50              # jumlah trial Optuna
RANDOM_SEED = 2025
MODEL_SAVE_PATH = "model_lgb_best.txt"

# -------------------------
# CHECK: features_df exist?
# -------------------------
if "features_df" not in globals():
    raise RuntimeError("Tidak menemukan `features_df`. Siapkan dataframe fitur (index datetime) dan kolom target 'pm25' sebelum menjalankan script ini.")

df = features_df.copy()
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Kolom target '{TARGET_COL}' tidak ditemukan di features_df. Sesuaikan TARGET_COL.")

# Pastikan data terurut (time index)
if not isinstance(df.index, pd.DatetimeIndex):
    raise RuntimeError("Index features_df harus berupa DatetimeIndex (waktu).")

df = df.sort_index()

# -------------------------
# Split train / holdout (last chunk sebagai test/holdout)
# -------------------------
n_total = len(df)
n_test = max(1, int(n_total * TEST_RATIO))
n_train = n_total - n_test

df_train = df.iloc[:n_train].copy()
df_test = df.iloc[n_train:].copy()

X_train_all = df_train.drop(columns=[TARGET_COL]).values
y_train_all = df_train[TARGET_COL].values

X_test = df_test.drop(columns=[TARGET_COL]).values
y_test = df_test[TARGET_COL].values

print(f"Total rows: {n_total}, train rows: {len(df_train)}, holdout rows: {len(df_test)}")

# -------------------------
# Helper: create rolling-origin fold indices
# expanding training window, fixed validation window
# -------------------------
def make_time_series_folds(n_splits: int, n_train_len: int, n_val_len: int = None):
    """
    Returns list of (train_idx, val_idx) tuples using expanding window.
    If n_val_len is None, we divide remaining training window into n_splits equal blocks as validation windows.
    """
    if n_val_len is None:
        # evenly split last part of train into n_splits blocks
        # define step as floor(n_train_len / (n_splits + 1))
        step = max(1, n_train_len // (n_splits + 1))
        folds = []
        for k in range(1, n_splits + 1):
            train_end = step * k
            val_start = train_end
            val_end = val_start + step
            if val_end > n_train_len:
                val_end = n_train_len
            train_idx = np.arange(0, train_end)
            val_idx = np.arange(val_start, val_end)
            if len(val_idx) == 0:
                break
            folds.append((train_idx, val_idx))
        return folds
    else:
        # fixed validation length: create folds with expanding train until there is space for val windows
        folds = []
        start_val = n_train_len - n_splits * n_val_len
        if start_val < 1:
            raise ValueError("Not enough data for the given n_splits and n_val_len.")
        for k in range(n_splits):
            train_idx = np.arange(0, start_val + k * n_val_len)
            val_idx = np.arange(start_val + k * n_val_len, start_val + (k + 1) * n_val_len)
            folds.append((train_idx, val_idx))
        return folds

# Choose folds using training length
n_train_len = len(df_train)
folds = make_time_series_folds(N_FOLDS, n_train_len)
print(f"Generated {len(folds)} folds. Example fold sizes (train,val):", [(len(t), len(v)) for t, v in folds])

# -------------------------
# Optuna objective
# -------------------------
def objective(trial):
    # define search space
    param = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": RANDOM_SEED,
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 5),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
    }
    # accumulate per-fold errors
    val_maes = []
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        X_tr = df_train.drop(columns=[TARGET_COL]).values[train_idx]
        y_tr = df_train[TARGET_COL].values[train_idx]
        X_val = df_train.drop(columns=[TARGET_COL]).values[val_idx]
        y_val = df_train[TARGET_COL].values[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dvalid = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        # train with early stopping; verbose = 0 to reduce prints
        model = lgb.train(
            param,
            dtrain,
            valid_sets=[dtrain, dvalid],
            valid_names=["train", "valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )


        # validation predictions (use best_iteration_)
        best_iter = model.best_iteration or param["n_estimators"]
        ypred = model.predict(X_val, num_iteration=best_iter)
        fold_mae = mean_absolute_error(y_val, ypred)
        val_maes.append(fold_mae)

        # report intermediate objective value to Optuna for pruning decisions
        trial.report(fold_mae, fold_idx)
        if trial.should_prune():
            raise TrialPruned()

    # return mean MAE across folds
    mean_mae = float(np.mean(val_maes))
    return mean_mae

# -------------------------
# Run Optuna study
# -------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
print("Starting Optuna tuning...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Tuning finished.")
print("Best MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# -------------------------
# Train final model on full training set with best params
# -------------------------
best_params = study.best_params.copy()
# ensure required entries
best_params.update({
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "seed": RANDOM_SEED
})

# Train final on full training DF (train_all)
X_train_full = df_train.drop(columns=[TARGET_COL]).values
y_train_full = df_train[TARGET_COL].values

dtrain_full = lgb.Dataset(X_train_full, label=y_train_full)

print("Training final model on full training set...")
model_final = lgb.train(
    best_params,
    dtrain_full,
    num_boost_round=best_params.get("n_estimators", 1000),
    callbacks=[lgb.log_evaluation(period=0)]
)

# Save model
model_final.save_model(MODEL_SAVE_PATH)
print(f"Final model saved to {MODEL_SAVE_PATH}")

# -------------------------
# Evaluate final model on holdout test
# -------------------------
y_pred_test = model_final.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = math.sqrt(mean_squared_error(y_test, y_pred_test))
mape = np.mean(np.abs((y_test - y_pred_test) / np.where(y_test==0, 1e-8, y_test))) * 100
r2 = r2_score(y_test, y_pred_test)

print("\n=== Final evaluation on holdout test ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")

# Optional: show last few predictions vs actual
df_eval = pd.DataFrame({
    "actual": y_test,
    "pred": y_pred_test
}, index=df_test.index)
print("\nLast 10 holdout rows (actual vs pred):")
print(df_eval.tail(10))

# Done
print("\nOptuna + TimeSeries CV tuning complete.")


c:\Users\Yusuf Rajabi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total rows: 11586, train rows: 3476, holdout rows: 8110
Generated 5 folds. Example fold sizes (train,val): [(579, 579), (1158, 579), (1737, 579), (2316, 579), (2895, 579)]
Starting Optuna tuning...


Best trial: 22. Best value: 5.34286: 100%|██████████| 50/50 [01:08<00:00,  1.37s/it]


Tuning finished.
Best MAE: 5.342858651692064
Best params:
  num_leaves: 162
  max_depth: 8
  learning_rate: 0.030811678127976666
  min_data_in_leaf: 27
  feature_fraction: 0.9567317873760987
  bagging_fraction: 0.9632184817123006
  bagging_freq: 1
  lambda_l1: 0.9022402774134036
  lambda_l2: 1.0534243220121529
  min_gain_to_split: 0.2682294660985136
  n_estimators: 2421
Training final model on full training set...
Final model saved to model_lgb_best.txt

=== Final evaluation on holdout test ===
MAE  : 6.6650
RMSE : 8.8399
MAPE : 24.43%
R²   : 0.9961

Last 10 holdout rows (actual vs pred):
                                  actual       pred
created_at                                         
2026-04-09 11:59:35.304375+00:00   12.77  29.688033
2026-04-09 12:00:36.523274+00:00   15.42  29.827238
2026-04-09 12:01:38.255833+00:00   13.83  30.004390
2026-04-09 12:02:39.058031+00:00   16.47  28.758786
2026-04-09 12:03:39.418037+00:00   16.47  29.381910
2026-04-09 12:04:39.254242+00:00   16.12